# 02 — Silver: Sellers + T10 trust score (M2)

**Owner:** M2 (was M6) • **Tier:** Heavy • **Phase:** 4 (Thu 14 May lunch – Sun 17 May)

> **Note:** ownership moved from M6 to M2 per the new allocation. T10 seller trust score requires defended weight choices, which is heavy-tier work. M2 also owns T4 (in `02_silver_customer.ipynb`) and T7 (formula belongs here OR in `02_silver_classified.ipynb`).

## Scope
- **Source tables you write Silver for here (3):** `ts_sellers`, `ts_seller_status_codes`, `ts_seller_types`
- **Silver transform owned: T10** — Seller trust score (composite of 8 inputs, mapped to 5 risk tiers)
- **Plus T7** — NL listing confidence score (formula here or in classified notebook)

## T10 — Seller trust score (suggested formula; defend weights)
```
score = 0.20 · (1 - cancellation_rate)
      + 0.15 · (1 - late_fulfilment_rate)
      + 0.15 · (1 - return_rate)
      + 0.10 · (1 - complaint_rate_per_100)
      + 0.10 · (1 - nl_duplicate_rate)
      + 0.10 · (1 - nl_report_rate)
      + 0.10 · response_rate
      − 0.10 · moderation_actions_normalized
```
Clamp to [0.0, 1.0]. Map to 5 tiers per dim_seller_risk_tier (Verified Trusted ≥0.85; Standard 0.65–0.85; Under Review 0.40–0.65; High Risk 0.20–0.40; Suspended <0.20).

**Equal weighting is unacceptable per brief.** Defend your weight choices in report Section 7.

## Anomaly territory (in your sellers slice)
- B8 (Seller trust score formula defence) — you defend in writing
- B6 (Estimated Classified GMV formula) — you defend in writing (lower/point/upper bands)

## Hands-on-of-everything checklist (your overall)
- [ ] T4 customer identity (in 02_silver_customer.ipynb) — your hardest piece
- [ ] T7 NL confidence (formula + defended weights)
- [ ] T10 seller trust score (here)
- [ ] dim_customer (SCD2 + identity bridge)
- [ ] dim_seller_risk_tier (5 tiers, static lookup but supports your T10 mapping)
- [ ] fact_seller_performance_snapshot (per seller per week, numerator/denominator pairs)
- [ ] Anomalies: A11 (9999 collision), B5, B6, B8
- [ ] Report: Sections 7 (T4/T7/T10), 8 (A11, B5, B6, B8), 9 (your dims/fact)


## Setup — Install connector + widgets (Free Edition serverless)

_Brief Section 7.4 specified a Spark-Snowflake Maven JAR; Free Edition replaces this with the pure-Python `snowflake-connector-python`. See `docs/databricks_setup.md`._

In [ ]:
%pip install -q snowflake-connector-python pandas rapidfuzz
# dbutils.library.restartPython()  # SKIP on Free Edition — kernel hangs

In [ ]:
dbutils.widgets.text('sf_account',   'rhxendw-yb24678')
dbutils.widgets.text('sf_user',      'NEXAMART_LEAD')   # change to NEXAMART_M{2..6} for members
dbutils.widgets.text('sf_password',  '')                # paste at notebook run time
dbutils.widgets.text('sf_warehouse', 'NEXAMART_WH')
dbutils.widgets.text('sf_role',      'ACCOUNTADMIN')    # NEXAMART_ENGINEER for members

## Cell 2 — Imports

In [ ]:
from pyspark.sql import functions as F, Window
import sys
sys.path.append('/Workspace/Repos/<your-org>/nexamart-m1/notebooks/_shared')
from utils_dates     import parse_date, is_parse_failure
from utils_keys      import surrogate_key, anonymous_key
from utils_anomaly   import add_anomaly_columns, flag, flag_date_parse_failure
from utils_snowflake import read_from_snowflake, write_to_snowflake

def read_bronze(t):
    return read_from_snowflake(spark, t, schema='NEXAMART_BRONZE')

def read_silver(t):
    return read_from_snowflake(spark, t, schema='NEXAMART_SILVER')

def write_silver(df, t):
    n = write_to_snowflake(df, t, schema='NEXAMART_SILVER')
    print(f'Wrote silver.{t} ({n} rows)')

def write_gold(df, t):
    n = write_to_snowflake(df, t, schema='NEXAMART_GOLD')
    print(f'Wrote gold.{t} ({n} rows)')

## Cell 3 — `silver_sellers` + `silver_seller_risk_signals` + `silver_seller_safety_reports`

**TODO:**
- Read `ts_sellers`, `ts_risk_signals`, `ts_safety_reports`
- Parse dates (iso_date + iso_timestamp formats)
- SK on `seller_id`, `signal_id`, `report_id`
- Apply T2 status normalisation on seller_status
- Write three Silver tables

In [ ]:
# TODO M2: seller silvers

## Cell 4 — `silver_reviews` + A15 detection

**TODO:**
- Read `rv_reviews`, parse `review_datetime` (iso_timestamp)
- SK on `review_id`
- Join `silver_customer_master` for customer FK
- **Flag A15:** rows where `days_post_delivery < 0` → `REVIEW_BEFORE_DELIVERY`, status `FLAGGED_ANOMALY`, certainty `UNRELIABLE`
- Cross-reference `is_verified_purchase` — note in your report Section 8 what pattern emerges (verified vs unverified for the negative-days subset)
- Write

In [ ]:
# TODO M2: reviews + A15

## Cell 5 — `silver_cases` + `silver_case_events` + A16 detection

**TODO:**
- Read `cs_cases` (parse `case_open_datetime` with `slash_datetime` format), `cs_case_events`
- SK on `case_id`, `(case_id, event_seq)`
- **Flag A16:** rows where `is_duplicate_flag = 1` → `DUPLICATE_CASE`, status `FLAGGED_ANOMALY`, certainty `INFERRED`
- Carry `canonical_case_ref` through to Silver — it becomes the dedup key for `fact_customer_complaint`
- Note: `cs_cases` channel attribution is manually entered (per UNDERSTANDING.md) — flag rows with implausible channel values as `MANUAL_CHANNEL_ATTRIBUTION` (FLAGGED_AMBIGUOUS) where appropriate
- Write both Silver tables

In [ ]:
# TODO M2: cases + A16

## Cell 6 — T10: `silver_seller_trust_score`

Compute the 8 input rates per seller, apply your weighted formula, map to risk tier.

**TODO:**
- For each seller, compute (from your other Silvers + Bronze):
  - `cancellation_rate` = cancelled marketplace orders / total marketplace orders
  - `late_fulfilment_rate` = fulfilment events past SLA / total fulfilments
  - `return_rate` = returns / orders
  - `complaint_rate` = cs_cases mentioning seller / orders
  - `nl_duplicate_rate` = your A12 detection / NL listings for this seller
  - `nl_report_rate` = ts_safety_reports about seller's listings / NL listings
  - `response_rate` = chats responded / chats received (if available)
  - `moderation_actions` = ts_risk_signals count
- Apply weighted composite per formula in Cell 1 markdown
- Map to 5 risk tiers (Verified Trusted / Standard / Under Review / High Risk / Suspended)
- Output `silver_seller_trust_score` with: seller_id (FK), trust_score, all 8 input metrics (numerators + denominators separately for non-additive integrity), risk_tier
- Flag sellers with `trust_score < 0.30` as `SELLER_HIGH_RISK`

**Document weights** in report Section 7. Equal weighting will be rejected.

In [ ]:
# TODO M2: T10 trust score

## Cell 7 — Acceptance check

Before PR review:
1. `silver_reviews` has flagged rows for `REVIEW_BEFORE_DELIVERY` (small count, all should be unverified)
2. `silver_cases` has `DUPLICATE_CASE` rows (handful)
3. `silver_seller_trust_score` exists with one row per seller, all 5 risk tiers populated (or noted if no sellers fall in one)
4. T10 formula and weight choices documented in report Section 7
5. All 4 mandatory anomaly columns populated